In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# ============================================================
# SHOPEE IMAGE MATCHING — FINAL pHash BASELINE
# ============================================================
#
# COMPLETE END-TO-END PIPELINE
#
# Original dataframe
#       ↓
# Train / Validation split
#       ↓
# Existing image_phash cleaning
#       ↓
# Validation positive + negative pairs
#       ↓
# Hamming distance
#       ↓
# pHash similarity
#       ↓
# 5-Fold CV
#       ↓
# Learn best threshold on training fold
#       ↓
# Evaluate on unseen validation fold
#       ↓
# OOF predictions
#       ↓
# Final threshold = median CV threshold
#       ↓
# Final metrics
#       ↓
# Error analysis
#       ↓
# Nearest-neighbour retrieval
#       ↓
# Save CSVs + plots
#
# IMPORTANT:
# pHash is ALREADY PRESENT in the dataframe.
# WE DO NOT GENERATE pHash.
#
# ============================================================


In [ ]:
#PART C APPROACH 1
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# INPUT DATA
# ------------------------------------------------------------

# Change this to your actual CSV path.
#
# Example:
# DATA_PATH = "train.csv"
#
# If you ALREADY have the complete dataframe loaded as
# `train_df`, you can comment out the pd.read_csv line
# and set:
#
# df = train_df.copy()
# ------------------------------------------------------------

DATA_PATH = "train.csv"


# ------------------------------------------------------------
# COLUMN NAMES
# ------------------------------------------------------------

PHASH_COLUMN = "image_phash"

PRODUCT_ID_COLUMN = "label_group"

IMAGE_ID_COLUMN = "posting_id"

IMAGE_COLUMN = "image"


# ------------------------------------------------------------
# TRAIN / VALIDATION CONFIGURATION
# ------------------------------------------------------------

VALIDATION_SIZE = 0.20

RANDOM_STATE = 42


# ------------------------------------------------------------
# PAIR GENERATION
# ------------------------------------------------------------

# Maximum number of positive pairs sampled from each product.
POSITIVE_PAIRS_PER_PRODUCT = 10

# Number of negative pairs.
NEGATIVE_PAIRS = 10000


# ------------------------------------------------------------
# CROSS VALIDATION
# ------------------------------------------------------------

N_SPLITS = 5


# ------------------------------------------------------------
# RETRIEVAL
# ------------------------------------------------------------

TOP_K = 10


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = "results/phash_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# RANDOM SEEDS
# ------------------------------------------------------------

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ============================================================
# 3. LOAD ORIGINAL DATAFRAME
# ============================================================

print("=" * 80)
print("LOADING DATA")
print("=" * 80)


# ------------------------------------------------------------
# OPTION A:
# Load from CSV
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)


# ------------------------------------------------------------
# If you already loaded the dataframe earlier, replace the
# previous line with:
#
# df = train_df.copy()
# ------------------------------------------------------------


print(f"Total original rows: {len(df):,}")

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 4. BASIC DATA VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("BASIC DATA VALIDATION")
print("=" * 80)


required_columns = [
    PHASH_COLUMN,
    PRODUCT_ID_COLUMN
]


for col in required_columns:

    if col not in df.columns:

        raise ValueError(
            f"\nRequired column '{col}' not found.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )


print("\nRequired columns found.")

print(
    f"Number of unique products: "
    f"{df[PRODUCT_ID_COLUMN].nunique():,}"
)


print("\nMissing values:")

print(
    df[
        required_columns
    ].isnull().sum()
)


# ============================================================
# 5. REMOVE ROWS WITHOUT PRODUCT LABEL
# ============================================================

df = df.dropna(
    subset=[PRODUCT_ID_COLUMN]
).reset_index(drop=True)


print(
    f"\nRows after removing missing product labels: "
    f"{len(df):,}"
)


# ============================================================
# 6. CREATE TRAIN / VALIDATION SPLIT
# ============================================================
#
# IMPORTANT:
#
# We need validation images from the SAME product so that
# positive image pairs can actually exist.
#
# A normal GroupShuffleSplit by label_group would put an
# entire product into one split and therefore produce zero
# positive validation pairs.
#
# Therefore we perform a PRODUCT-WISE INTERNAL SPLIT:
#
# Product A:
#
#   images → some train
#          → some validation
#
# Product B:
#
#   images → some train
#          → some validation
#
# This gives validation_df images from the same product,
# allowing positive pairs.
# ============================================================

print("\n" + "=" * 80)
print("CREATING TRAIN / VALIDATION SPLIT")
print("=" * 80)


train_parts = []
validation_parts = []

rng = np.random.RandomState(RANDOM_STATE)


for product_id, group in df.groupby(
    PRODUCT_ID_COLUMN,
    sort=False
):

    group = group.copy()

    n = len(group)

    # Shuffle indices for this product
    shuffled_indices = rng.permutation(
        group.index.to_numpy()
    )

    # --------------------------------------------------------
    # Products with only one image:
    #
    # They cannot produce positive pairs.
    # Keep them in train so they do not pollute validation.
    # --------------------------------------------------------

    if n == 1:

        train_parts.append(group)

        continue


    # --------------------------------------------------------
    # Products with 2 images:
    #
    # Put one image into validation and one into train.
    #
    # This product will NOT produce a positive validation pair.
    # --------------------------------------------------------

    if n == 2:

        val_count = 1


    # --------------------------------------------------------
    # Products with >=3 images:
    #
    # Ensure at least TWO validation images so that the product
    # contributes at least one positive pair.
    # --------------------------------------------------------

    else:

        val_count = max(
            2,
            int(round(n * VALIDATION_SIZE))
        )

        # Keep at least one training image
        val_count = min(
            val_count,
            n - 1
        )


    val_indices = shuffled_indices[
        :val_count
    ]

    train_indices = shuffled_indices[
        val_count:
    ]


    if len(train_indices) > 0:

        train_parts.append(
            group.loc[train_indices]
        )


    if len(val_indices) > 0:

        validation_parts.append(
            group.loc[val_indices]
        )


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

train_df = pd.concat(
    train_parts,
    ignore_index=True
)

validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Shuffle
# ------------------------------------------------------------

train_df = train_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


validation_df = validation_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


print(
    f"\nTrain samples      : {len(train_df):,}"
)

print(
    f"Validation samples : {len(validation_df):,}"
)

print(
    f"Train products     : "
    f"{train_df[PRODUCT_ID_COLUMN].nunique():,}"
)

print(
    f"Validation products: "
    f"{validation_df[PRODUCT_ID_COLUMN].nunique():,}"
)


# ============================================================
# 7. SAVE TRAIN / VALIDATION SPLITS
# ============================================================

train_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "train_split.csv"
    ),
    index=False
)

validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_split.csv"
    ),
    index=False
)


# ============================================================
# 8. CLEAN EXISTING pHASH
# ============================================================
#
# We DO NOT calculate pHash from images.
#
# The dataset already gives:
#
#       image_phash
#
# We simply convert its representation into an integer so that
# XOR and bit_count() can be used to calculate Hamming distance.
# ============================================================

print("\n" + "=" * 80)
print("CLEANING EXISTING pHASH VALUES")
print("=" * 80)


def phash_to_int(value):

    """
    Convert an existing pHash value into an integer.

    Supported:
        hexadecimal strings
        0x-prefixed hexadecimal
        binary strings
        Python integers
        NumPy integers
    """

    if pd.isna(value):

        return np.nan


    # Already integer
    if isinstance(
        value,
        (int, np.integer)
    ):

        return int(value)


    value = str(value).strip()


    if value == "":

        return np.nan


    # --------------------------------------------------------
    # Try hexadecimal
    # --------------------------------------------------------

    try:

        if value.lower().startswith("0x"):

            return int(
                value,
                16
            )

        return int(
            value,
            16
        )

    except ValueError:

        pass


    # --------------------------------------------------------
    # Try binary
    # --------------------------------------------------------

    try:

        return int(
            value,
            2
        )

    except ValueError:

        return np.nan


# ------------------------------------------------------------
# Work only on validation dataframe
# ------------------------------------------------------------

validation_df = validation_df.copy()


validation_df["phash_int"] = (
    validation_df[
        PHASH_COLUMN
    ]
    .apply(phash_to_int)
)


before = len(validation_df)


validation_df = validation_df.dropna(
    subset=[
        "phash_int",
        PRODUCT_ID_COLUMN
    ]
).reset_index(drop=True)


after = len(validation_df)


print(
    f"Invalid/missing pHash rows removed: "
    f"{before - after:,}"
)

print(
    f"Usable validation rows: "
    f"{after:,}"
)


# ============================================================
# 9. VERIFY 64-BIT pHASH
# ============================================================

MAX_PHASH = (
    (1 << 64) - 1
)


invalid_64bit = validation_df[
    validation_df[
        "phash_int"
    ].apply(
        lambda x:
        x < 0 or x > MAX_PHASH
    )
]


if len(invalid_64bit) > 0:

    print(
        f"\nWARNING: "
        f"{len(invalid_64bit):,} "
        "pHashes outside 64-bit range."
    )


    validation_df = validation_df[
        validation_df[
            "phash_int"
        ].apply(
            lambda x:
            0 <= x <= MAX_PHASH
        )
    ].reset_index(drop=True)


else:

    print(
        "\nAll pHashes are valid 64-bit values."
    )


# ============================================================
# 10. HAMMING DISTANCE
# ============================================================

def hamming_distance(
    hash1,
    hash2
):

    """
    Hamming distance between two 64-bit hashes.

    XOR identifies the bit positions that differ.

    bit_count() counts those differing bits.

    Example:

        hash1 = 101100
        hash2 = 100110

        XOR   = 001010

        distance = 2
    """

    return (
        int(hash1)
        ^
        int(hash2)
    ).bit_count()


def phash_similarity(
    distance
):

    """
    Convert Hamming distance into [0,1] similarity.

        distance = 0
            similarity = 1

        distance = 64
            similarity = 0
    """

    return (
        1.0
        -
        distance / 64.0
    )


# ============================================================
# 11. CONSTRUCT POSITIVE PAIRS
# ============================================================
#
# Positive pair:
#
#       same label_group
#
# Therefore:
#
#       y = 1
#
# ============================================================

print("\n" + "=" * 80)
print("CONSTRUCTING POSITIVE PAIRS")
print("=" * 80)


positive_pairs = []


for product_id, group in validation_df.groupby(
    PRODUCT_ID_COLUMN
):

    indices = group.index.tolist()


    # Need at least two images
    if len(indices) < 2:

        continue


    # Generate combinations
    possible_pairs = []


    for i in range(
        len(indices)
    ):

        for j in range(
            i + 1,
            len(indices)
        ):

            possible_pairs.append(
                (
                    indices[i],
                    indices[j]
                )
            )


    # --------------------------------------------------------
    # Limit pairs per product
    # --------------------------------------------------------

    if (
        len(possible_pairs)
        >
        POSITIVE_PAIRS_PER_PRODUCT
    ):

        possible_pairs = random.sample(
            possible_pairs,
            POSITIVE_PAIRS_PER_PRODUCT
        )


    # --------------------------------------------------------
    # Store pairs
    # --------------------------------------------------------

    for idx1, idx2 in possible_pairs:

        positive_pairs.append({

            "idx1": idx1,

            "idx2": idx2,

            "label": 1

        })


positive_pairs = pd.DataFrame(
    positive_pairs
)


print(
    f"Positive pairs generated: "
    f"{len(positive_pairs):,}"
)


# ============================================================
# 12. CONSTRUCT NEGATIVE PAIRS
# ============================================================
#
# Negative pair:
#
#       label_group_1 != label_group_2
#
# Therefore:
#
#       y = 0
#
# ============================================================

print("\n" + "=" * 80)
print("CONSTRUCTING NEGATIVE PAIRS")
print("=" * 80)


n = len(validation_df)


# ------------------------------------------------------------
# Maximum possible unique negative pairs
# ------------------------------------------------------------

product_array = validation_df[
    PRODUCT_ID_COLUMN
].values


total_possible_negative_pairs = 0


for i in range(n):

    for j in range(
        i + 1,
        n
    ):

        if product_array[i] != product_array[j]:

            total_possible_negative_pairs += 1


target_negative_pairs = min(
    NEGATIVE_PAIRS,
    total_possible_negative_pairs
)


print(
    f"Requested negative pairs: "
    f"{NEGATIVE_PAIRS:,}"
)

print(
    f"Possible negative pairs: "
    f"{total_possible_negative_pairs:,}"
)

print(
    f"Generating: "
    f"{target_negative_pairs:,}"
)


if target_negative_pairs == 0:

    raise ValueError(
        "No possible negative pairs. "
        "Validation set must contain at least two products."
    )


negative_pairs_set = set()


# ------------------------------------------------------------
# Random sampling
# ------------------------------------------------------------

while len(
    negative_pairs_set
) < target_negative_pairs:

    idx1 = random.randrange(n)

    idx2 = random.randrange(n)


    if idx1 == idx2:

        continue


    product1 = validation_df.iloc[
        idx1
    ][PRODUCT_ID_COLUMN]


    product2 = validation_df.iloc[
        idx2
    ][PRODUCT_ID_COLUMN]


    # Must be different products
    if product1 == product2:

        continue


    pair = tuple(
        sorted(
            (
                idx1,
                idx2
            )
        )
    )


    negative_pairs_set.add(
        pair
    )


negative_pairs = pd.DataFrame(
    [
        {
            "idx1": idx1,
            "idx2": idx2,
            "label": 0
        }

        for idx1, idx2
        in negative_pairs_set
    ]
)


print(
    f"Negative pairs generated: "
    f"{len(negative_pairs):,}"
)


# ============================================================
# 13. COMBINE POSITIVE + NEGATIVE PAIRS
# ============================================================

pairs_df = pd.concat(
    [
        positive_pairs,
        negative_pairs
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Shuffle
# ------------------------------------------------------------

pairs_df = pairs_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


print(
    f"\nTotal pairs: "
    f"{len(pairs_df):,}"
)


print("\nPair distribution:")

print(
    pairs_df[
        "label"
    ].value_counts()
)


# ============================================================
# 14. ADD IMAGE / PRODUCT INFORMATION TO PAIRS
# ============================================================
#
# This makes the saved pair files interpretable.
# ============================================================

def get_value(
    row,
    index,
    column
):

    if column in validation_df.columns:

        return validation_df.iloc[
            index
        ][column]

    return None


pairs_df["image_id_1"] = pairs_df[
    "idx1"
].apply(
    lambda x:
    get_value(
        None,
        int(x),
        IMAGE_ID_COLUMN
    )
    if IMAGE_ID_COLUMN in validation_df.columns
    else None
)


pairs_df["image_id_2"] = pairs_df[
    "idx2"
].apply(
    lambda x:
    get_value(
        None,
        int(x),
        IMAGE_ID_COLUMN
    )
    if IMAGE_ID_COLUMN in validation_df.columns
    else None
)


pairs_df["product_id_1"] = pairs_df[
    "idx1"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][PRODUCT_ID_COLUMN]
)


pairs_df["product_id_2"] = pairs_df[
    "idx2"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][PRODUCT_ID_COLUMN]
)


# ============================================================
# 15. COMPUTE HAMMING DISTANCE
# ============================================================

print("\n" + "=" * 80)
print("COMPUTING HAMMING DISTANCES")
print("=" * 80)


phash_array = (
    validation_df[
        "phash_int"
    ]
    .astype(np.uint64)
    .values
)


distances = []


for _, row in pairs_df.iterrows():

    idx1 = int(
        row["idx1"]
    )

    idx2 = int(
        row["idx2"]
    )


    distance = hamming_distance(
        phash_array[idx1],
        phash_array[idx2]
    )


    distances.append(
        distance
    )


pairs_df[
    "hamming_distance"
] = distances


# ============================================================
# 16. CONVERT HAMMING DISTANCE TO SIMILARITY
# ============================================================

pairs_df[
    "similarity"
] = (
    1.0
    -
    pairs_df[
        "hamming_distance"
    ] / 64.0
)


print("\nDistance statistics:")

print(
    pairs_df.groupby(
        "label"
    )[
        "hamming_distance"
    ].describe()
)


print("\nSimilarity statistics:")

print(
    pairs_df.groupby(
        "label"
    )[
        "similarity"
    ].describe()
)


# ============================================================
# 17. SAVE PAIR DATASET
# ============================================================

pairs_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_pairs.csv"
    ),
    index=False
)


# ============================================================
# 18. THRESHOLD RANGE
# ============================================================
#
# Hamming distance can only be:
#
#       0 ... 64
#
#
# Decision:
#
# distance <= threshold
#             ↓
#           MATCH
#
#
# distance > threshold
#             ↓
#         NON-MATCH
#
# ============================================================

THRESHOLDS = np.arange(
    0,
    65
)


# ============================================================
# 19. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    distances,
    threshold
):

    y_true = np.asarray(
        y_true
    )

    distances = np.asarray(
        distances
    )


    # --------------------------------------------------------
    # Prediction rule
    # --------------------------------------------------------

    y_pred = (
        distances <= threshold
    ).astype(int)


    # --------------------------------------------------------
    # Standard metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    # --------------------------------------------------------
    # Confusion matrix
    #
    #       predicted
    #
    #          0     1
    #
    # actual 0 TN    FP
    # actual 1 FN    TP
    #
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    return {

        "threshold": threshold,

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "specificity": specificity,

        "f1": f1,

        "tp": tp,

        "tn": tn,

        "fp": fp,

        "fn": fn
    }


# ============================================================
# 20. FIND BEST THRESHOLD
# ============================================================
#
# We DO NOT arbitrarily say:
#
#       threshold = 10
#
# Instead:
#
# For every possible threshold:
#
#       calculate F1
#
# Then select:
#
#       threshold = argmax(F1)
#
# ============================================================

def find_best_threshold(
    y_true,
    distances
):

    results = []


    for threshold in THRESHOLDS:

        metrics = calculate_metrics(
            y_true,
            distances,
            threshold
        )

        results.append(
            metrics
        )


    results_df = pd.DataFrame(
        results
    )


    # --------------------------------------------------------
    # Select threshold with maximum F1
    # --------------------------------------------------------

    best_idx = results_df[
        "f1"
    ].idxmax()


    best_result = results_df.loc[
        best_idx
    ]


    best_threshold = int(
        best_result[
            "threshold"
        ]
    )


    return (
        best_threshold,
        results_df
    )


# ============================================================
# 21. 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================
#
# IMPORTANT:
#
# Each fold performs:
#
#        TRAINING PAIR FOLD
#                 ↓
#        search threshold 0...64
#                 ↓
#          best F1 threshold
#                 ↓
#        HELD-OUT PAIR FOLD
#                 ↓
#             evaluate
#
# Therefore the threshold is NOT learned from the validation
# fold being evaluated.
#
# ============================================================

print("\n" + "=" * 80)
print("5-FOLD CROSS-VALIDATION")
print("=" * 80)


X = pairs_df[
    [
        "hamming_distance",
        "similarity"
    ]
]


y = pairs_df[
    "label"
].values


skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


fold_results = []

all_oof_predictions = []

all_threshold_results = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    skf.split(X, y),
    start=1
):


    print("\n" + "-" * 80)

    print(
        f"FOLD {fold}/{N_SPLITS}"
    )

    print("-" * 80)


    # --------------------------------------------------------
    # Split pairs
    # --------------------------------------------------------

    train_data = pairs_df.iloc[
        train_idx
    ]


    val_data = pairs_df.iloc[
        val_idx
    ]


    y_train = train_data[
        "label"
    ].values


    d_train = train_data[
        "hamming_distance"
    ].values


    y_val = val_data[
        "label"
    ].values


    d_val = val_data[
        "hamming_distance"
    ].values


    # --------------------------------------------------------
    # Learn threshold ONLY on training fold
    # --------------------------------------------------------

    best_threshold, threshold_df = (
        find_best_threshold(
            y_train,
            d_train
        )
    )


    threshold_df[
        "fold"
    ] = fold


    all_threshold_results.append(
        threshold_df
    )


    print(
        f"Best training threshold: "
        f"{best_threshold}"
    )


    # --------------------------------------------------------
    # Evaluate on unseen validation fold
    # --------------------------------------------------------

    fold_metrics = calculate_metrics(
        y_val,
        d_val,
        best_threshold
    )


    fold_metrics[
        "fold"
    ] = fold


    fold_results.append(
        fold_metrics
    )


    # --------------------------------------------------------
    # Out-of-fold predictions
    # --------------------------------------------------------

    val_predictions = val_data.copy()


    val_predictions[
        "fold"
    ] = fold


    val_predictions[
        "best_threshold"
    ] = best_threshold


    val_predictions[
        "prediction"
    ] = (
        val_predictions[
            "hamming_distance"
        ]
        <= best_threshold
    ).astype(int)


    all_oof_predictions.append(
        val_predictions
    )


    # --------------------------------------------------------
    # Print fold metrics
    # --------------------------------------------------------

    print(
        f"Accuracy    : "
        f"{fold_metrics['accuracy']:.4f}"
    )

    print(
        f"Precision   : "
        f"{fold_metrics['precision']:.4f}"
    )

    print(
        f"Recall      : "
        f"{fold_metrics['recall']:.4f}"
    )

    print(
        f"Specificity : "
        f"{fold_metrics['specificity']:.4f}"
    )

    print(
        f"F1          : "
        f"{fold_metrics['f1']:.4f}"
    )


# ============================================================
# 22. COMBINE CV RESULTS
# ============================================================

fold_results_df = pd.DataFrame(
    fold_results
)


threshold_results_all = pd.concat(
    all_threshold_results,
    ignore_index=True
)


oof_predictions = pd.concat(
    all_oof_predictions,
    ignore_index=True
)


# ============================================================
# 23. DISPLAY CV RESULTS
# ============================================================

print("\n" + "=" * 80)
print("CROSS-VALIDATION RESULTS")
print("=" * 80)


print(
    fold_results_df[
        [
            "fold",
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "specificity",
            "f1"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 24. CV SUMMARY
# ============================================================

cv_summary = {

    "accuracy_mean":
        fold_results_df[
            "accuracy"
        ].mean(),

    "accuracy_std":
        fold_results_df[
            "accuracy"
        ].std(),

    "precision_mean":
        fold_results_df[
            "precision"
        ].mean(),

    "precision_std":
        fold_results_df[
            "precision"
        ].std(),

    "recall_mean":
        fold_results_df[
            "recall"
        ].mean(),

    "recall_std":
        fold_results_df[
            "recall"
        ].std(),

    "specificity_mean":
        fold_results_df[
            "specificity"
        ].mean(),

    "specificity_std":
        fold_results_df[
            "specificity"
        ].std(),

    "f1_mean":
        fold_results_df[
            "f1"
        ].mean(),

    "f1_std":
        fold_results_df[
            "f1"
        ].std(),

    "threshold_mean":
        fold_results_df[
            "threshold"
        ].mean(),

    "threshold_std":
        fold_results_df[
            "threshold"
        ].std(),

    "threshold_median":
        fold_results_df[
            "threshold"
        ].median()
}


cv_summary_df = pd.DataFrame(
    [cv_summary]
)


print("\nCV mean ± std:")

print(
    f"Accuracy    : "
    f"{cv_summary['accuracy_mean']:.4f} "
    f"+/- "
    f"{cv_summary['accuracy_std']:.4f}"
)

print(
    f"Precision   : "
    f"{cv_summary['precision_mean']:.4f} "
    f"+/- "
    f"{cv_summary['precision_std']:.4f}"
)

print(
    f"Recall      : "
    f"{cv_summary['recall_mean']:.4f} "
    f"+/- "
    f"{cv_summary['recall_std']:.4f}"
)

print(
    f"Specificity : "
    f"{cv_summary['specificity_mean']:.4f} "
    f"+/- "
    f"{cv_summary['specificity_std']:.4f}"
)

print(
    f"F1          : "
    f"{cv_summary['f1_mean']:.4f} "
    f"+/- "
    f"{cv_summary['f1_std']:.4f}"
)

print(
    f"Threshold   : "
    f"{cv_summary['threshold_mean']:.2f} "
    f"+/- "
    f"{cv_summary['threshold_std']:.2f}"
)


# ============================================================
# 25. FINAL THRESHOLD
# ============================================================
#
# Each fold produces one optimal threshold:
#
#       T1
#       T2
#       T3
#       T4
#       T5
#
# Final threshold:
#
#       median(T1,T2,T3,T4,T5)
#
# Median is robust to an unusual fold.
# ============================================================

FINAL_THRESHOLD = int(
    np.median(
        fold_results_df[
            "threshold"
        ].values
    )
)


FINAL_SIMILARITY_THRESHOLD = (
    phash_similarity(
        FINAL_THRESHOLD
    )
)


print("\n" + "=" * 80)
print("FINAL THRESHOLD")
print("=" * 80)


print(
    f"Fold thresholds: "
    f"{fold_results_df['threshold'].tolist()}"
)


print(
    f"Final Hamming threshold: "
    f"{FINAL_THRESHOLD}"
)


print(
    f"Equivalent similarity threshold: "
    f"{FINAL_SIMILARITY_THRESHOLD:.4f}"
)


# ============================================================
# 26. FINAL OOF EVALUATION
# ============================================================
#
# Apply ONE final threshold to all OOF pairs.
#
# This gives a single overall OOF evaluation.
# ============================================================

oof_y_true = oof_predictions[
    "label"
].values


oof_distances = oof_predictions[
    "hamming_distance"
].values


oof_pred = (
    oof_distances
    <= FINAL_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# Standard metrics
# ------------------------------------------------------------

final_accuracy = accuracy_score(
    oof_y_true,
    oof_pred
)


final_precision = precision_score(
    oof_y_true,
    oof_pred,
    zero_division=0
)


final_recall = recall_score(
    oof_y_true,
    oof_pred,
    zero_division=0
)


final_f1 = f1_score(
    oof_y_true,
    oof_pred,
    zero_division=0
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    oof_y_true,
    oof_pred,
    labels=[0, 1]
).ravel()


final_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)


# ============================================================
# 27. ROC-AUC / PR-AUC
# ============================================================
#
# For ROC/PR:
#
# higher similarity = more likely match
#
# similarity = 1 - distance/64
# ============================================================

oof_similarity = (
    1.0
    -
    oof_distances / 64.0
)


roc_auc = roc_auc_score(
    oof_y_true,
    oof_similarity
)


pr_auc = average_precision_score(
    oof_y_true,
    oof_similarity
)


# ============================================================
# 28. FINAL METRIC SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL OOF RESULTS")
print("=" * 80)


print(
    f"Final Hamming Threshold : "
    f"{FINAL_THRESHOLD}"
)

print(
    f"Similarity Threshold    : "
    f"{FINAL_SIMILARITY_THRESHOLD:.4f}"
)

print(
    f"Accuracy                : "
    f"{final_accuracy:.4f}"
)

print(
    f"Precision               : "
    f"{final_precision:.4f}"
)

print(
    f"Recall                  : "
    f"{final_recall:.4f}"
)

print(
    f"Specificity             : "
    f"{final_specificity:.4f}"
)

print(
    f"F1                      : "
    f"{final_f1:.4f}"
)

print(
    f"ROC-AUC                 : "
    f"{roc_auc:.4f}"
)

print(
    f"PR-AUC                  : "
    f"{pr_auc:.4f}"
)


# ============================================================
# 29. ADD FINAL PREDICTIONS TO OOF DATA
# ============================================================

oof_predictions[
    "final_prediction"
] = oof_pred


oof_predictions[
    "final_similarity"
] = oof_similarity


oof_predictions[
    "correct"
] = (
    oof_predictions[
        "label"
    ]
    ==
    oof_predictions[
        "final_prediction"
    ]
)


# ============================================================
# 30. DISTRIBUTION PLOT
# ============================================================
#
# Compare Hamming distances for:
#
#       Positive pairs
#       Negative pairs
#
# Expected:
#
# Positive → smaller distances
# Negative → larger distances
# ============================================================

plt.figure(
    figsize=(10, 6)
)


sns.histplot(
    data=pairs_df,
    x="hamming_distance",
    hue="label",
    bins=33,
    kde=True,
    stat="density",
    common_norm=False
)


plt.axvline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Threshold = "
        f"{FINAL_THRESHOLD}"
    )
)


plt.title(
    "pHash Hamming Distance Distribution"
)


plt.xlabel(
    "Hamming Distance"
)


plt.ylabel(
    "Density"
)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "hamming_distance_distribution.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 31. SIMILARITY DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(10, 6)
)


sns.histplot(
    data=pairs_df,
    x="similarity",
    hue="label",
    bins=33,
    kde=True,
    stat="density",
    common_norm=False
)


plt.axvline(
    FINAL_SIMILARITY_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Threshold = "
        f"{FINAL_SIMILARITY_THRESHOLD:.3f}"
    )
)


plt.title(
    "pHash Similarity Distribution"
)


plt.xlabel(
    "Similarity"
)


plt.ylabel(
    "Density"
)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "similarity_distribution.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 32. THRESHOLD vs F1
# ============================================================

mean_threshold_results = (
    threshold_results_all
    .groupby("threshold")
    .agg(
        mean_f1=("f1", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_specificity=("specificity", "mean")
    )
    .reset_index()
)


plt.figure(
    figsize=(10, 6)
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "mean_f1"
    ],
    marker="o",
    label="F1"
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "mean_accuracy"
    ],
    label="Accuracy"
)


plt.axvline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Final threshold = "
        f"{FINAL_THRESHOLD}"
    )
)


plt.xlabel(
    "Hamming Distance Threshold"
)


plt.ylabel(
    "Score"
)


plt.title(
    "Threshold Selection"
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "threshold_selection.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 33. THRESHOLD STABILITY ACROSS FOLDS
# ============================================================

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    fold_results_df[
        "fold"
    ],
    fold_results_df[
        "threshold"
    ],
    marker="o"
)


plt.axhline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Median threshold = "
        f"{FINAL_THRESHOLD}"
    )
)


plt.xlabel(
    "Fold"
)


plt.ylabel(
    "Best Hamming Threshold"
)


plt.title(
    "Threshold Stability Across Folds"
)


plt.xticks(
    fold_results_df[
        "fold"
    ]
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "threshold_stability.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 34. ROC CURVE
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(
    oof_y_true,
    oof_similarity
)


plt.figure(
    figsize=(7, 7)
)


plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=(
        f"pHash "
        f"(AUC = {roc_auc:.4f})"
    )
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)


plt.xlabel(
    "False Positive Rate"
)


plt.ylabel(
    "True Positive Rate"
)


plt.title(
    "ROC Curve — pHash"
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "roc_curve.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 35. PRECISION-RECALL CURVE
# ============================================================

precision_curve, recall_curve, pr_thresholds = (
    precision_recall_curve(
        oof_y_true,
        oof_similarity
    )
)


plt.figure(
    figsize=(7, 7)
)


plt.plot(
    recall_curve,
    precision_curve,
    linewidth=2,
    label=(
        f"PR-AUC = "
        f"{pr_auc:.4f}"
    )
)


plt.xlabel(
    "Recall"
)


plt.ylabel(
    "Precision"
)


plt.title(
    "Precision-Recall Curve — pHash"
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "precision_recall_curve.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 36. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    oof_y_true,
    oof_pred,
    labels=[0, 1]
)


plt.figure(
    figsize=(6, 5)
)


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Non-Match",
        "Match"
    ],
    yticklabels=[
        "Non-Match",
        "Match"
    ]
)


plt.xlabel(
    "Predicted"
)


plt.ylabel(
    "Actual"
)


plt.title(
    "Confusion Matrix — pHash"
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "confusion_matrix.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 37. RETRIEVAL / NEAREST NEIGHBOURS
# ============================================================
#
# Given query image:
#
#       query pHash
#             ↓
#       compare against every image
#             ↓
#       Hamming distance
#             ↓
#       sort ascending
#             ↓
#       top-K neighbours
#
# The query itself is excluded.
# ============================================================

def get_phash_nearest_neighbors(
    dataframe,
    query_idx,
    top_k=10
):

    query_hash = dataframe.loc[
        query_idx,
        "phash_int"
    ]


    records = []


    for idx in dataframe.index:

        # Do not retrieve itself
        if idx == query_idx:

            continue


        candidate_hash = dataframe.loc[
            idx,
            "phash_int"
        ]


        distance = hamming_distance(
            query_hash,
            candidate_hash
        )


        similarity = phash_similarity(
            distance
        )


        records.append({

            "query_idx": query_idx,

            "neighbor_idx": idx,

            "hamming_distance": distance,

            "similarity": similarity,

            "query_product": dataframe.loc[
                query_idx,
                PRODUCT_ID_COLUMN
            ],

            "neighbor_product": dataframe.loc[
                idx,
                PRODUCT_ID_COLUMN
            ]

        })


    neighbors = pd.DataFrame(
        records
    )


    neighbors = neighbors.sort_values(
        "hamming_distance",
        ascending=True
    )


    return neighbors.head(
        top_k
    ).reset_index(
        drop=True
    )


# ============================================================
# 38. RETRIEVAL METRICS
# ============================================================
#
# For each query:
#
# Top-1:
#     Is nearest image from same product?
#
# Top-K:
#     Does ANY of top-K belong to same product?
# ============================================================

def retrieval_metrics(
    dataframe,
    top_k=10,
    max_queries=None
):

    query_indices = list(
        dataframe.index
    )


    if max_queries is not None:

        query_indices = query_indices[
            :max_queries
        ]


    top1_correct = 0

    topk_correct = 0

    evaluated = 0


    for query_idx in query_indices:

        query_product = dataframe.loc[
            query_idx,
            PRODUCT_ID_COLUMN
        ]


        neighbors = get_phash_nearest_neighbors(
            dataframe,
            query_idx,
            top_k=top_k
        )


        if len(neighbors) == 0:

            continue


        evaluated += 1


        # ----------------------------------------------------
        # Top-1
        # ----------------------------------------------------

        top1_product = neighbors.iloc[
            0
        ][
            "neighbor_product"
        ]


        if top1_product == query_product:

            top1_correct += 1


        # ----------------------------------------------------
        # Top-K
        # ----------------------------------------------------

        if query_product in neighbors[
            "neighbor_product"
        ].values:

            topk_correct += 1


    top1_accuracy = (
        top1_correct / evaluated
        if evaluated > 0
        else 0
    )


    topk_accuracy = (
        topk_correct / evaluated
        if evaluated > 0
        else 0
    )


    return {

        "queries": evaluated,

        "top_1_accuracy":
            top1_accuracy,

        f"top_{top_k}_hit_rate":
            topk_accuracy
    }


# ------------------------------------------------------------
# Calculate retrieval
# ------------------------------------------------------------

retrieval_result = retrieval_metrics(
    validation_df,
    top_k=TOP_K
)


print("\n" + "=" * 80)
print("NEAREST-NEIGHBOUR RETRIEVAL")
print("=" * 80)


print(
    f"Queries evaluated: "
    f"{retrieval_result['queries']:,}"
)


print(
    f"Top-1 Accuracy: "
    f"{retrieval_result['top_1_accuracy']:.4f}"
)


print(
    f"Top-{TOP_K} Hit Rate: "
    f"{retrieval_result[f'top_{TOP_K}_hit_rate']:.4f}"
)


# ============================================================
# 39. SAMPLE NEAREST-NEIGHBOUR QUERY
# ============================================================

sample_query_idx = validation_df.index[0]


sample_neighbors = get_phash_nearest_neighbors(
    validation_df,
    sample_query_idx,
    top_k=TOP_K
)


print("\nExample query:")


print(
    validation_df.loc[
        sample_query_idx
    ]
)


print("\nNearest neighbours:")


print(
    sample_neighbors
)


# ============================================================
# 40. ERROR ANALYSIS
# ============================================================
#
# False Positive:
#
# actual = 0
# predicted = 1
#
# Different products look visually similar according to pHash.
#
#
# False Negative:
#
# actual = 1
# predicted = 0
#
# Same product has sufficiently different visual appearance.
#
# ============================================================

print("\n" + "=" * 80)
print("ERROR ANALYSIS")
print("=" * 80)


false_positives = oof_predictions[
    (
        oof_predictions[
            "label"
        ] == 0
    )
    &
    (
        oof_predictions[
            "final_prediction"
        ] == 1
    )
].copy()


false_negatives = oof_predictions[
    (
        oof_predictions[
            "label"
        ] == 1
    )
    &
    (
        oof_predictions[
            "final_prediction"
        ] == 0
    )
].copy()


print(
    f"False Positives: "
    f"{len(false_positives):,}"
)


print(
    f"False Negatives: "
    f"{len(false_negatives):,}"
)


# ============================================================
# 41. MOST CONFIDENT FALSE POSITIVES
# ============================================================
#
# For false positives:
#
# Smaller distance = stronger visual similarity.
#
# Therefore sort ascending.
# ============================================================

false_positives = false_positives.sort_values(
    "hamming_distance",
    ascending=True
)


print("\nMost confident False Positives:")


fp_columns = [
    "idx1",
    "idx2",
    "image_id_1",
    "image_id_2",
    "product_id_1",
    "product_id_2",
    "hamming_distance",
    "similarity",
    "label",
    "final_prediction"
]


fp_columns = [
    col
    for col in fp_columns
    if col in false_positives.columns
]


print(
    false_positives[
        fp_columns
    ].head(20).to_string(
        index=False
    )
)


# ============================================================
# 42. MOST DIFFICULT FALSE NEGATIVES
# ============================================================
#
# For false negatives:
#
# Larger distance = same-product images that look very
# different according to pHash.
#
# Therefore sort descending.
# ============================================================

false_negatives = false_negatives.sort_values(
    "hamming_distance",
    ascending=False
)


print("\nMost difficult False Negatives:")


fn_columns = [
    "idx1",
    "idx2",
    "image_id_1",
    "image_id_2",
    "product_id_1",
    "product_id_2",
    "hamming_distance",
    "similarity",
    "label",
    "final_prediction"
]


fn_columns = [
    col
    for col in fn_columns
    if col in false_negatives.columns
]


print(
    false_negatives[
        fn_columns
    ].head(20).to_string(
        index=False
    )
)


# ============================================================
# 43. SAVE ALL RESULTS
# ============================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)


# ------------------------------------------------------------
# Fold results
# ------------------------------------------------------------

fold_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "kfold_results.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# CV summary
# ------------------------------------------------------------

cv_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cv_summary.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Threshold search
# ------------------------------------------------------------

threshold_results_all.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_results_all_folds.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# OOF predictions
# ------------------------------------------------------------

oof_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "oof_predictions.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Errors
# ------------------------------------------------------------

false_positives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "false_positives.csv"
    ),
    index=False
)


false_negatives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "false_negatives.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Nearest neighbours
# ------------------------------------------------------------

sample_neighbors.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sample_nearest_neighbors.csv"
    ),
    index=False
)


# ============================================================
# 44. FINAL CONFIGURATION FILE
# ============================================================

configuration = pd.DataFrame({

    "parameter": [

        "method",

        "representation",

        "distance_measure",

        "similarity_measure",

        "number_of_folds",

        "validation_size",

        "positive_pairs_per_product",

        "negative_pairs",

        "final_hamming_threshold",

        "final_similarity_threshold",

        "roc_auc",

        "pr_auc",

        "accuracy",

        "precision",

        "recall",

        "specificity",

        "f1",

        "top_1_accuracy",

        f"top_{TOP_K}_hit_rate"

    ],


    "value": [

        "pHash",

        "Existing Shopee image_phash",

        "Hamming distance",

        "1 - HammingDistance / 64",

        N_SPLITS,

        VALIDATION_SIZE,

        POSITIVE_PAIRS_PER_PRODUCT,

        target_negative_pairs,

        FINAL_THRESHOLD,

        FINAL_SIMILARITY_THRESHOLD,

        roc_auc,

        pr_auc,

        final_accuracy,

        final_precision,

        final_recall,

        final_specificity,

        final_f1,

        retrieval_result[
            "top_1_accuracy"
        ],

        retrieval_result[
            f"top_{TOP_K}_hit_rate"
        ]

    ]
})


configuration.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phash_configuration.csv"
    ),
    index=False
)


# ============================================================
# 45. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL pHash BASELINE SUMMARY")
print("=" * 80)


print(
    f"""
DATA
----
Original samples:
    {len(df):,}

Training samples:
    {len(train_df):,}

Validation samples:
    {len(validation_df):,}

Validation products:
    {validation_df[PRODUCT_ID_COLUMN].nunique():,}


PAIR DATASET
------------
Positive pairs:
    {len(positive_pairs):,}

Negative pairs:
    {len(negative_pairs):,}

Total pairs:
    {len(pairs_df):,}


REPRESENTATION
--------------
Representation:
    Existing Shopee pHash

pHash generation:
    NOT PERFORMED

pHash size:
    64 bits


DISTANCE / SIMILARITY
---------------------
Distance:
    Hamming distance

Similarity:
    1 - HammingDistance / 64


CROSS-VALIDATION
----------------
Folds:
    {N_SPLITS}

Threshold selection:
    Maximum F1 on training fold

Fold thresholds:
    {fold_results_df['threshold'].tolist()}

Final threshold:
    {FINAL_THRESHOLD}

Similarity threshold:
    {FINAL_SIMILARITY_THRESHOLD:.4f}


FINAL OOF PERFORMANCE
---------------------
Accuracy:
    {final_accuracy:.4f}

Precision:
    {final_precision:.4f}

Recall:
    {final_recall:.4f}

Specificity:
    {final_specificity:.4f}

F1:
    {final_f1:.4f}

ROC-AUC:
    {roc_auc:.4f}

PR-AUC:
    {pr_auc:.4f}


RETRIEVAL
---------
Top-1 Accuracy:
    {retrieval_result['top_1_accuracy']:.4f}

Top-{TOP_K} Hit Rate:
    {retrieval_result[f'top_{TOP_K}_hit_rate']:.4f}


OUTPUT DIRECTORY
----------------
{OUTPUT_DIR}
"""
)


# ============================================================
# 46. LIST OUTPUT FILES
# ============================================================

print("\nGenerated files:")


for filename in sorted(
    os.listdir(
        OUTPUT_DIR
    )
):

    print(
        "  -",
        filename
    )


print("\n" + "=" * 80)
print("pHash PIPELINE COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# SHOPEE PRICE MATCHING PART C
# APPROACH 2: PRETRAINED RESNET50 IMAGE EMBEDDINGS
# ============================================================
#
# COMPLETE END-TO-END PIPELINE
#
# DATAFRAME
#    |
#    +--> TRAIN_DF
#    |
#    +--> VALIDATION_DF
#             |
#             +--> POSITIVE PAIRS
#             |
#             +--> NEGATIVE PAIRS
#                      |
#                      v
#              RESNET50 EMBEDDINGS
#                      |
#                      v
#              COSINE SIMILARITY
#                      |
#                      v
#              K-FOLD CV
#                      |
#             threshold selection
#                      |
#                      v
#              OOF PREDICTIONS
#                      |
#          +-----------+-----------+
#          |           |           |
#        Metrics      ROC/PR     Retrieval
#          |
#          v
#      Error Analysis
#
#
# IMPORTANT:
# - image_phash is NOT used here.
# - We use the actual image files.
# - ResNet50 is pretrained and frozen.
# - No CNN weights are trained.
# - K-fold CV is used to learn the cosine-similarity threshold.
#
# ============================================================


# ============================================================
# 1. INSTALLATION NOTE
# ============================================================
#
# If needed:
#
# pip install torch torchvision pandas numpy scikit-learn
# pip install matplotlib seaborn pillow tqdm
#
# ============================================================


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    balanced_accuracy_score,
    matthews_corrcoef
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)


warnings.filterwarnings("ignore")


# ============================================================
# 3. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# CHANGE THESE TWO PATHS
# ------------------------------------------------------------

DATA_PATH = "train.csv"

IMAGE_DIR = "train_images"


# ------------------------------------------------------------
# DATAFRAME COLUMNS
# ------------------------------------------------------------

IMAGE_COLUMN = "image"

IMAGE_ID_COLUMN = "posting_id"

PRODUCT_ID_COLUMN = "label_group"


# ------------------------------------------------------------
# TRAIN / VALIDATION
# ------------------------------------------------------------

VALIDATION_SIZE = 0.20

RANDOM_STATE = 42


# ------------------------------------------------------------
# PAIR GENERATION
# ------------------------------------------------------------

POSITIVE_PAIRS_PER_PRODUCT = 10

NEGATIVE_PAIRS = 10000


# ------------------------------------------------------------
# CROSS VALIDATION
# ------------------------------------------------------------

N_SPLITS = 5


# ------------------------------------------------------------
# RESNET CONFIGURATION
# ------------------------------------------------------------

IMAGE_SIZE = 224

BATCH_SIZE = 64

NUM_WORKERS = 0

# Set to True if you have CUDA.
# The code automatically checks availability anyway.
PIN_MEMORY = True


# ------------------------------------------------------------
# RETRIEVAL
# ------------------------------------------------------------

TOP_K = 10


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = "results/resnet50_final"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 4. RANDOM SEEDS
# ============================================================

random.seed(
    RANDOM_STATE
)

np.random.seed(
    RANDOM_STATE
)

torch.manual_seed(
    RANDOM_STATE
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_STATE
    )


# ============================================================
# 5. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 90)

print("DEVICE")

print("=" * 90)

print(
    f"Using device: {DEVICE}"
)


if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )


# ============================================================
# 6. LOAD THE ORIGINAL DATAFRAME
# ============================================================

print("\n" + "=" * 90)

print("LOADING ORIGINAL DATAFRAME")

print("=" * 90)


df = pd.read_csv(
    DATA_PATH
)


print(
    f"Total rows: {len(df):,}"
)


print("\nColumns:")

print(
    df.columns.tolist()
)


# ============================================================
# 7. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    IMAGE_COLUMN,
    PRODUCT_ID_COLUMN
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"\nMissing required column: {column}\n"
            f"Available columns:\n{df.columns.tolist()}"
        )


print(
    "\nRequired columns found."
)


# ============================================================
# 8. BASIC CLEANING
# ============================================================

print("\n" + "=" * 90)

print("BASIC DATA CLEANING")

print("=" * 90)


# Remove rows without image filename
df = df.dropna(
    subset=[
        IMAGE_COLUMN
    ]
).copy()


# Remove rows without product label
df = df.dropna(
    subset=[
        PRODUCT_ID_COLUMN
    ]
).copy()


# Remove duplicate posting IDs if applicable
if IMAGE_ID_COLUMN in df.columns:

    df = df.drop_duplicates(
        subset=[
            IMAGE_ID_COLUMN
        ]
    )


df = df.reset_index(
    drop=True
)


print(
    f"Rows after cleaning: {len(df):,}"
)


print(
    f"Unique products: "
    f"{df[PRODUCT_ID_COLUMN].nunique():,}"
)


# ============================================================
# 9. RESOLVE IMAGE PATHS
# ============================================================
#
# The dataframe normally contains:
#
#       image = "1000.jpg"
#
# and the actual image is:
#
#       train_images/1000.jpg
#
# ============================================================

def resolve_image_path(
    image_name
):

    image_name = str(
        image_name
    )


    # If dataframe already contains an absolute path
    if os.path.isabs(
        image_name
    ):

        return image_name


    return os.path.join(
        IMAGE_DIR,
        image_name
    )


df["image_path"] = (
    df[IMAGE_COLUMN]
    .apply(
        resolve_image_path
    )
)


# ============================================================
# 10. CHECK IMAGE FILES
# ============================================================

print("\n" + "=" * 90)

print("CHECKING IMAGE FILES")

print("=" * 90)


df["image_exists"] = (
    df["image_path"]
    .apply(
        os.path.isfile
    )
)


missing_images = (
    ~df["image_exists"]
).sum()


print(
    f"Images found   : "
    f"{len(df) - missing_images:,}"
)

print(
    f"Images missing : "
    f"{missing_images:,}"
)


if missing_images > 0:

    print(
        "\nFirst missing image paths:"
    )

    print(
        df.loc[
            ~df["image_exists"],
            "image_path"
        ].head(20).to_string(
            index=False
        )
    )


    # Remove missing images
    df = df[
        df["image_exists"]
    ].copy()


df = df.drop(
    columns=[
        "image_exists"
    ]
).reset_index(
    drop=True
)


if len(df) == 0:

    raise ValueError(
        "No valid images found. "
        "Check DATA_PATH and IMAGE_DIR."
    )


# ============================================================
# 11. CREATE TRAINING + VALIDATION DATAFRAMES
# ============================================================
#
# IMPORTANT:
#
# We want validation images from the same product to allow
# positive image pairs.
#
# Therefore we split INSIDE each label_group.
#
# Example:
#
# Product A:
#
#       A1 A2 A3 A4 A5
#
# becomes:
#
# train:
#       A1 A3 A5
#
# validation:
#       A2 A4
#
# This allows:
#
#       (A2, A4) -> positive pair
#
# ============================================================

print("\n" + "=" * 90)

print("CREATING TRAINING AND VALIDATION DATAFRAMES")

print("=" * 90)


rng = np.random.RandomState(
    RANDOM_STATE
)


train_parts = []

validation_parts = []


for product_id, group in df.groupby(
    PRODUCT_ID_COLUMN,
    sort=False
):

    group = group.copy()


    n = len(group)


    # --------------------------------------------------------
    # One image:
    #
    # Cannot form a positive validation pair.
    # Keep in training.
    # --------------------------------------------------------

    if n == 1:

        train_parts.append(
            group
        )

        continue


    indices = group.index.to_numpy()


    shuffled = rng.permutation(
        indices
    )


    # --------------------------------------------------------
    # Two images:
    #
    # One train + one validation.
    #
    # This product won't contribute a positive validation pair,
    # but it remains available for negative pairs.
    # --------------------------------------------------------

    if n == 2:

        val_count = 1


    # --------------------------------------------------------
    # >= 3 images:
    #
    # Ensure at least 2 validation images.
    # --------------------------------------------------------

    else:

        val_count = max(
            2,
            int(
                round(
                    n * VALIDATION_SIZE
                )
            )
        )


        # Always leave at least one training image
        val_count = min(
            val_count,
            n - 1
        )


    validation_indices = (
        shuffled[
            :val_count
        ]
    )


    training_indices = (
        shuffled[
            val_count:
        ]
    )


    if len(training_indices) > 0:

        train_parts.append(
            group.loc[
                training_indices
            ]
        )


    if len(validation_indices) > 0:

        validation_parts.append(
            group.loc[
                validation_indices
            ]
        )


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

train_df = pd.concat(
    train_parts,
    ignore_index=True
)


validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Shuffle
# ------------------------------------------------------------

train_df = train_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(
    drop=True
)


validation_df = validation_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(
    drop=True
)


print(
    f"\nOriginal dataframe:"
    f" {len(df):,}"
)


print(
    f"Training dataframe:"
    f" {len(train_df):,}"
)


print(
    f"Validation dataframe:"
    f" {len(validation_df):,}"
)


print(
    f"Training products:"
    f" {train_df[PRODUCT_ID_COLUMN].nunique():,}"
)


print(
    f"Validation products:"
    f" {validation_df[PRODUCT_ID_COLUMN].nunique():,}"
)


# ============================================================
# 12. SAVE TRAINING / VALIDATION DATAFRAMES
# ============================================================

train_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "train_df.csv"
    ),
    index=False
)


validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_df.csv"
    ),
    index=False
)


df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "complete_dataframe.csv"
    ),
    index=False
)


# ============================================================
# 13. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 90)

print("DATASET SUMMARY")

print("=" * 90)


print(
    "\nTraining label distribution:"
)


print(
    train_df[
        PRODUCT_ID_COLUMN
    ].value_counts().describe()
)


print(
    "\nValidation label distribution:"
)


print(
    validation_df[
        PRODUCT_ID_COLUMN
    ].value_counts().describe()
)


# ============================================================
# 14. BUILD POSITIVE PAIRS
# ============================================================
#
# Positive:
#
#       label_group_1 == label_group_2
#
#       pair label = 1
#
# ============================================================

print("\n" + "=" * 90)

print("CREATING POSITIVE IMAGE PAIRS")

print("=" * 90)


positive_pairs = []


for product_id, group in validation_df.groupby(
    PRODUCT_ID_COLUMN
):

    indices = group.index.tolist()


    if len(indices) < 2:

        continue


    # --------------------------------------------------------
    # Randomly sample pair combinations without constructing
    # an unnecessarily huge list when possible.
    # --------------------------------------------------------

    possible_pairs = []


    for i in range(
        len(indices)
    ):

        for j in range(
            i + 1,
            len(indices)
        ):

            possible_pairs.append(
                (
                    indices[i],
                    indices[j]
                )
            )


    if len(possible_pairs) > POSITIVE_PAIRS_PER_PRODUCT:

        possible_pairs = random.sample(
            possible_pairs,
            POSITIVE_PAIRS_PER_PRODUCT
        )


    for idx1, idx2 in possible_pairs:

        positive_pairs.append({

            "idx1": idx1,

            "idx2": idx2,

            "label": 1

        })


positive_pairs = pd.DataFrame(
    positive_pairs
)


print(
    f"Positive pairs: "
    f"{len(positive_pairs):,}"
)


# ============================================================
# 15. BUILD NEGATIVE PAIRS
# ============================================================
#
# Negative:
#
#       label_group_1 != label_group_2
#
#       pair label = 0
#
# ============================================================

print("\n" + "=" * 90)

print("CREATING NEGATIVE IMAGE PAIRS")

print("=" * 90)


n = len(validation_df)


product_labels = (
    validation_df[
        PRODUCT_ID_COLUMN
    ].values
)


negative_pair_set = set()


while len(
    negative_pair_set
) < NEGATIVE_PAIRS:

    idx1 = random.randrange(
        n
    )

    idx2 = random.randrange(
        n
    )


    if idx1 == idx2:

        continue


    if (
        product_labels[idx1]
        ==
        product_labels[idx2]
    ):

        continue


    pair = tuple(
        sorted(
            [
                idx1,
                idx2
            ]
        )
    )


    negative_pair_set.add(
        pair
    )


negative_pairs = pd.DataFrame(
    [
        {
            "idx1": idx1,
            "idx2": idx2,
            "label": 0
        }

        for idx1, idx2
        in negative_pair_set
    ]
)


print(
    f"Negative pairs: "
    f"{len(negative_pairs):,}"
)


# ============================================================
# 16. COMBINE PAIR DATAFRAME
# ============================================================

pairs_df = pd.concat(
    [
        positive_pairs,
        negative_pairs
    ],
    ignore_index=True
)


pairs_df = pairs_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(
    drop=True
)


# ============================================================
# 17. ADD HUMAN-READABLE PAIR INFORMATION
# ============================================================

if IMAGE_ID_COLUMN in validation_df.columns:

    pairs_df["image_id_1"] = pairs_df[
        "idx1"
    ].apply(
        lambda x:
        validation_df.iloc[
            int(x)
        ][IMAGE_ID_COLUMN]
    )


    pairs_df["image_id_2"] = pairs_df[
        "idx2"
    ].apply(
        lambda x:
        validation_df.iloc[
            int(x)
        ][IMAGE_ID_COLUMN]
    )


pairs_df["image_1"] = pairs_df[
    "idx1"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][IMAGE_COLUMN]
)


pairs_df["image_2"] = pairs_df[
    "idx2"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][IMAGE_COLUMN]
)


pairs_df["image_path_1"] = pairs_df[
    "idx1"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ]["image_path"]
)


pairs_df["image_path_2"] = pairs_df[
    "idx2"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ]["image_path"]
)


pairs_df["product_id_1"] = pairs_df[
    "idx1"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][PRODUCT_ID_COLUMN]
)


pairs_df["product_id_2"] = pairs_df[
    "idx2"
].apply(
    lambda x:
    validation_df.iloc[
        int(x)
    ][PRODUCT_ID_COLUMN]
)


print("\nPair dataframe:")

print(
    pairs_df.head()
)


print(
    f"\nTotal pairs: "
    f"{len(pairs_df):,}"
)


print(
    "\nPair class distribution:"
)


print(
    pairs_df[
        "label"
    ].value_counts()
)


# ============================================================
# 18. SAVE PAIR DATAFRAME
# ============================================================

pairs_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_pairs.csv"
    ),
    index=False
)


# ============================================================
# 19. PYTORCH IMAGE DATASET
# ============================================================

weights = (
    ResNet50_Weights.DEFAULT
)


transform = (
    weights.transforms()
)


class ShopeeImageDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]


        image_path = row[
            "image_path"
        ]


        try:

            image = Image.open(
                image_path
            ).convert(
                "RGB"
            )

        except Exception as e:

            raise RuntimeError(
                f"Could not load image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )


        if self.transform is not None:

            image = self.transform(
                image
            )


        return (
            image,
            index
        )


# ============================================================
# 20. LOAD PRETRAINED RESNET50
# ============================================================
#
# We remove the final classification layer.
#
# Original:
#
#       image
#         ↓
#       ResNet50
#         ↓
#       2048-dimensional feature
#         ↓
#       FC classifier
#
# We use:
#
#       image
#         ↓
#       ResNet50
#         ↓
#       2048-dimensional embedding
#
# ============================================================

print("\n" + "=" * 90)

print("LOADING PRETRAINED RESNET50")

print("=" * 90)


resnet_model = resnet50(
    weights=weights
)


# Remove ImageNet classification layer
resnet_model.fc = nn.Identity()


resnet_model = resnet_model.to(
    DEVICE
)


# ------------------------------------------------------------
# FREEZE EVERYTHING
# ------------------------------------------------------------

resnet_model.eval()


for parameter in resnet_model.parameters():

    parameter.requires_grad = False


print(
    "ResNet50 loaded."
)


print(
    "Classification head removed."
)


print(
    "CNN parameters frozen."
)


print(
    "Embedding dimension = 2048"
)


# ============================================================
# 21. EMBEDDING EXTRACTION FUNCTION
# ============================================================

@torch.no_grad()
def extract_embeddings(
    dataframe,
    model,
    batch_size=BATCH_SIZE
):

    dataset = ShopeeImageDataset(
        dataframe,
        transform=transform
    )


    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            PIN_MEMORY
            and DEVICE.type == "cuda"
        )
    )


    embeddings = []

    indices = []


    for images, batch_indices in tqdm(
        loader,
        desc="Extracting ResNet50 embeddings"
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        features = model(
            images
        )


        # ----------------------------------------------------
        # L2 normalize embeddings
        #
        # This makes cosine similarity equivalent to dot
        # product.
        # ----------------------------------------------------

        features = torch.nn.functional.normalize(
            features,
            p=2,
            dim=1
        )


        embeddings.append(
            features.cpu().numpy()
        )


        indices.append(
            batch_indices.numpy()
        )


    embeddings = np.concatenate(
        embeddings,
        axis=0
    )


    indices = np.concatenate(
        indices,
        axis=0
    )


    # --------------------------------------------------------
    # Ensure original order
    # --------------------------------------------------------

    ordering = np.argsort(
        indices
    )


    embeddings = embeddings[
        ordering
    ]


    return embeddings


# ============================================================
# 22. EXTRACT VALIDATION EMBEDDINGS
# ============================================================
#
# IMPORTANT:
#
# We only need validation embeddings for the matching
# experiment.
#
# The ResNet50 weights are NOT trained on our Shopee data.
#
# ============================================================

print("\n" + "=" * 90)

print("EXTRACTING VALIDATION EMBEDDINGS")

print("=" * 90)


validation_embeddings = extract_embeddings(
    validation_df,
    resnet_model
)


print(
    f"\nEmbedding matrix shape:"
)

print(
    validation_embeddings.shape
)


print(
    f"Expected:"
    f" ({len(validation_df)}, 2048)"
)


# ============================================================
# 23. SAVE EMBEDDINGS
# ============================================================

np.save(
    os.path.join(
        OUTPUT_DIR,
        "validation_resnet50_embeddings.npy"
    ),
    validation_embeddings
)


# ============================================================
# 24. VERIFY NORMALIZATION
# ============================================================

embedding_norms = np.linalg.norm(
    validation_embeddings,
    axis=1
)


print(
    "\nEmbedding norm statistics:"
)


print(
    pd.Series(
        embedding_norms
    ).describe()
)


# ============================================================
# 25. COSINE SIMILARITY
# ============================================================
#
# For normalized embeddings:
#
#       cosine_similarity(a,b)
#
# is simply:
#
#       a dot b
#
#
# Range:
#
#       [-1, 1]
#
# In practice for image embeddings the relevant matching
# region is generally positive.
#
# ============================================================

def cosine_similarity(
    embedding1,
    embedding2
):

    return float(
        np.dot(
            embedding1,
            embedding2
        )
    )


# ============================================================
# 26. COMPUTE PAIR COSINE SIMILARITIES
# ============================================================

print("\n" + "=" * 90)

print("COMPUTING COSINE SIMILARITIES FOR ALL PAIRS")

print("=" * 90)


pair_similarities = []


for _, row in tqdm(
    pairs_df.iterrows(),
    total=len(pairs_df),
    desc="Pair similarity"
):

    idx1 = int(
        row["idx1"]
    )

    idx2 = int(
        row["idx2"]
    )


    similarity = cosine_similarity(
        validation_embeddings[idx1],
        validation_embeddings[idx2]
    )


    pair_similarities.append(
        similarity
    )


pairs_df[
    "cosine_similarity"
] = pair_similarities


# ============================================================
# 27. SIMILARITY STATISTICS
# ============================================================

print("\n" + "=" * 90)

print("COSINE SIMILARITY STATISTICS")

print("=" * 90)


print(
    pairs_df.groupby(
        "label"
    )[
        "cosine_similarity"
    ].describe()
)


# ============================================================
# 28. SAVE PAIR DATAFRAME WITH SIMILARITIES
# ============================================================

pairs_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_pairs_with_resnet_similarity.csv"
    ),
    index=False
)


# ============================================================
# 29. THRESHOLD RANGE
# ============================================================
#
# We don't arbitrarily choose:
#
#       0.5
#       0.7
#       0.8
#
# Instead, threshold is learned from training folds.
#
# Cosine similarity lies between -1 and 1.
#
# We search a dense range.
# ============================================================

THRESHOLDS = np.linspace(
    -1.0,
    1.0,
    401
)


# ============================================================
# 30. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    similarities,
    threshold
):

    y_true = np.asarray(
        y_true
    )


    similarities = np.asarray(
        similarities
    )


    # --------------------------------------------------------
    # Higher cosine similarity = more likely MATCH
    # --------------------------------------------------------

    y_pred = (
        similarities >= threshold
    ).astype(int)


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )


    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0
    )


    return {

        "threshold":
            threshold,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "specificity":
            specificity,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "f1":
            f1,

        "mcc":
            mcc,

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn
    }


# ============================================================
# 31. FIND BEST THRESHOLD
# ============================================================
#
# Objective:
#
#       maximize F1
#
# on the TRAINING fold.
#
# ============================================================

def find_best_threshold(
    y_true,
    similarities
):

    results = []


    for threshold in THRESHOLDS:

        metrics = calculate_metrics(
            y_true,
            similarities,
            threshold
        )


        results.append(
            metrics
        )


    results_df = pd.DataFrame(
        results
    )


    # --------------------------------------------------------
    # Maximize F1
    # --------------------------------------------------------

    best_idx = results_df[
        "f1"
    ].idxmax()


    best_threshold = float(
        results_df.loc[
            best_idx,
            "threshold"
        ]
    )


    return (
        best_threshold,
        results_df
    )


# ============================================================
# 32. K-FOLD CROSS VALIDATION
# ============================================================
#
# IMPORTANT:
#
# We are NOT training ResNet50.
#
# K-fold CV is used for:
#
#       threshold selection + robust evaluation
#
#
# Each fold:
#
#      TRAIN PAIRS
#          ↓
#      find best cosine threshold
#          ↓
#      HELD-OUT PAIRS
#          ↓
#      calculate metrics
#
# ============================================================

print("\n" + "=" * 90)

print("5-FOLD CROSS-VALIDATION")

print("=" * 90)


X = pairs_df[
    [
        "cosine_similarity"
    ]
].values


y = pairs_df[
    "label"
].values


skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


fold_results = []

all_threshold_results = []

all_oof_predictions = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    skf.split(
        X,
        y
    ),
    start=1
):

    print("\n" + "-" * 90)

    print(
        f"FOLD {fold}/{N_SPLITS}"
    )

    print("-" * 90)


    # --------------------------------------------------------
    # Pair training fold
    # --------------------------------------------------------

    fold_train = pairs_df.iloc[
        train_idx
    ]


    # --------------------------------------------------------
    # Pair validation fold
    # --------------------------------------------------------

    fold_val = pairs_df.iloc[
        val_idx
    ]


    y_train = fold_train[
        "label"
    ].values


    s_train = fold_train[
        "cosine_similarity"
    ].values


    y_val = fold_val[
        "label"
    ].values


    s_val = fold_val[
        "cosine_similarity"
    ].values


    # --------------------------------------------------------
    # Learn threshold ONLY from training fold
    # --------------------------------------------------------

    best_threshold, threshold_df = (
        find_best_threshold(
            y_train,
            s_train
        )
    )


    threshold_df[
        "fold"
    ] = fold


    all_threshold_results.append(
        threshold_df
    )


    print(
        f"Best training threshold:"
        f" {best_threshold:.4f}"
    )


    # --------------------------------------------------------
    # Evaluate on held-out fold
    # --------------------------------------------------------

    metrics = calculate_metrics(
        y_val,
        s_val,
        best_threshold
    )


    metrics[
        "fold"
    ] = fold


    fold_results.append(
        metrics
    )


    # --------------------------------------------------------
    # OOF predictions
    # --------------------------------------------------------

    fold_predictions = fold_val.copy()


    fold_predictions[
        "fold"
    ] = fold


    fold_predictions[
        "best_threshold"
    ] = best_threshold


    fold_predictions[
        "prediction"
    ] = (
        fold_predictions[
            "cosine_similarity"
        ]
        >= best_threshold
    ).astype(int)


    all_oof_predictions.append(
        fold_predictions
    )


    # --------------------------------------------------------
    # Print fold metrics
    # --------------------------------------------------------

    print(
        f"Accuracy           : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Balanced Accuracy  : "
        f"{metrics['balanced_accuracy']:.4f}"
    )

    print(
        f"Precision          : "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Recall             : "
        f"{metrics['recall']:.4f}"
    )

    print(
        f"Specificity        : "
        f"{metrics['specificity']:.4f}"
    )

    print(
        f"FPR                : "
        f"{metrics['fpr']:.4f}"
    )

    print(
        f"FNR                : "
        f"{metrics['fnr']:.4f}"
    )

    print(
        f"F1                 : "
        f"{metrics['f1']:.4f}"
    )

    print(
        f"MCC                : "
        f"{metrics['mcc']:.4f}"
    )


# ============================================================
# 33. COMBINE CV RESULTS
# ============================================================

fold_results_df = pd.DataFrame(
    fold_results
)


threshold_results_df = pd.concat(
    all_threshold_results,
    ignore_index=True
)


oof_predictions = pd.concat(
    all_oof_predictions,
    ignore_index=True
)


# ============================================================
# 34. PRINT K-FOLD RESULTS
# ============================================================

print("\n" + "=" * 90)

print("K-FOLD RESULTS")

print("=" * 90)


print(
    fold_results_df[
        [
            "fold",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "fpr",
            "fnr",
            "f1",
            "mcc"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 35. FINAL THRESHOLD
# ============================================================
#
# Final threshold = median of fold-optimal thresholds.
#
# ============================================================

FINAL_THRESHOLD = float(
    np.median(
        fold_results_df[
            "threshold"
        ].values
    )
)


print("\n" + "=" * 90)

print("FINAL COSINE-SIMILARITY THRESHOLD")

print("=" * 90)


print(
    "Fold thresholds:"
)


print(
    fold_results_df[
        "threshold"
    ].round(4).tolist()
)


print(
    f"\nFinal threshold: "
    f"{FINAL_THRESHOLD:.4f}"
)


# ============================================================
# 36. FINAL OOF PREDICTIONS
# ============================================================

oof_y_true = oof_predictions[
    "label"
].values


oof_similarity = oof_predictions[
    "cosine_similarity"
].values


oof_prediction = (
    oof_similarity
    >= FINAL_THRESHOLD
).astype(int)


oof_predictions[
    "final_prediction"
] = oof_prediction


oof_predictions[
    "correct"
] = (
    oof_y_true
    ==
    oof_prediction
)


# ============================================================
# 37. FINAL CLASSIFICATION METRICS
# ============================================================

final_accuracy = accuracy_score(
    oof_y_true,
    oof_prediction
)


final_balanced_accuracy = (
    balanced_accuracy_score(
        oof_y_true,
        oof_prediction
    )
)


final_precision = precision_score(
    oof_y_true,
    oof_prediction,
    zero_division=0
)


final_recall = recall_score(
    oof_y_true,
    oof_prediction,
    zero_division=0
)


final_f1 = f1_score(
    oof_y_true,
    oof_prediction,
    zero_division=0
)


final_mcc = matthews_corrcoef(
    oof_y_true,
    oof_prediction
)


# ============================================================
# 38. CONFUSION MATRIX METRICS
# ============================================================

tn, fp, fn, tp = confusion_matrix(
    oof_y_true,
    oof_prediction,
    labels=[
        0,
        1
    ]
).ravel()


final_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)


final_fpr = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0
)


final_fnr = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0
)


# ============================================================
# 39. ROC-AUC
# ============================================================

roc_auc = roc_auc_score(
    oof_y_true,
    oof_similarity
)


# ============================================================
# 40. PR-AUC
# ============================================================

pr_auc = average_precision_score(
    oof_y_true,
    oof_similarity
)


# ============================================================
# 41. FINAL METRICS TABLE
# ============================================================

final_metrics = pd.DataFrame({

    "metric": [

        "Accuracy",

        "Balanced Accuracy",

        "Precision",

        "Recall",

        "Specificity",

        "False Positive Rate",

        "False Negative Rate",

        "F1",

        "MCC",

        "ROC-AUC",

        "PR-AUC",

        "TP",

        "TN",

        "FP",

        "FN"

    ],


    "value": [

        final_accuracy,

        final_balanced_accuracy,

        final_precision,

        final_recall,

        final_specificity,

        final_fpr,

        final_fnr,

        final_f1,

        final_mcc,

        roc_auc,

        pr_auc,

        tp,

        tn,

        fp,

        fn

    ]

})


print("\n" + "=" * 90)

print("FINAL RESNET50 RESULTS")

print("=" * 90)


for _, row in final_metrics.iterrows():

    if row["metric"] in [
        "TP",
        "TN",
        "FP",
        "FN"
    ]:

        print(
            f"{row['metric']:22s}: "
            f"{int(row['value'])}"
        )

    else:

        print(
            f"{row['metric']:22s}: "
            f"{row['value']:.4f}"
        )


print(
    f"\nFinal threshold: "
    f"{FINAL_THRESHOLD:.4f}"
)


# ============================================================
# 42. CV MEAN / STD
# ============================================================

cv_metric_names = [

    "accuracy",

    "balanced_accuracy",

    "precision",

    "recall",

    "specificity",

    "fpr",

    "fnr",

    "f1",

    "mcc"

]


cv_summary_rows = []


for metric in cv_metric_names:

    cv_summary_rows.append({

        "metric": metric,

        "mean":
            fold_results_df[
                metric
            ].mean(),

        "std":
            fold_results_df[
                metric
            ].std()

    })


cv_summary_df = pd.DataFrame(
    cv_summary_rows
)


print("\n" + "=" * 90)

print("CROSS-VALIDATION MEAN ± STD")

print("=" * 90)


for _, row in cv_summary_df.iterrows():

    print(
        f"{row['metric']:22s}: "
        f"{row['mean']:.4f} "
        f"+/- "
        f"{row['std']:.4f}"
    )


# ============================================================
# 43. SAVE METRICS
# ============================================================

final_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_metrics.csv"
    ),
    index=False
)


fold_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "kfold_results.csv"
    ),
    index=False
)


cv_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cv_summary.csv"
    ),
    index=False
)


threshold_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_results_all_folds.csv"
    ),
    index=False
)


oof_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "oof_predictions.csv"
    ),
    index=False
)


# ============================================================
# 44. PLOT 1 — POSITIVE VS NEGATIVE SIMILARITY
# ============================================================

plt.figure(
    figsize=(10, 6)
)


sns.histplot(
    data=pairs_df,
    x="cosine_similarity",
    hue="label",
    bins=50,
    kde=True,
    stat="density",
    common_norm=False
)


plt.axvline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Threshold = "
        f"{FINAL_THRESHOLD:.3f}"
    )
)


plt.title(
    "ResNet50 Cosine Similarity Distribution"
)


plt.xlabel(
    "Cosine Similarity"
)


plt.ylabel(
    "Density"
)


plt.legend(
    title="Class"
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "01_similarity_distribution.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 45. PLOT 2 — BOXPLOT
# ============================================================

plt.figure(
    figsize=(9, 6)
)


sns.boxplot(
    data=pairs_df,
    x="label",
    y="cosine_similarity"
)


plt.xticks(
    [
        0,
        1
    ],
    [
        "Different Product",
        "Same Product"
    ]
)


plt.xlabel(
    "Ground Truth"
)


plt.ylabel(
    "Cosine Similarity"
)


plt.title(
    "ResNet50 Similarity by Ground Truth"
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "02_similarity_boxplot.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 46. PLOT 3 — THRESHOLD VS METRICS
# ============================================================

mean_threshold_results = (
    threshold_results_df
    .groupby(
        "threshold"
    )
    .agg(
        accuracy=(
            "accuracy",
            "mean"
        ),
        balanced_accuracy=(
            "balanced_accuracy",
            "mean"
        ),
        precision=(
            "precision",
            "mean"
        ),
        recall=(
            "recall",
            "mean"
        ),
        specificity=(
            "specificity",
            "mean"
        ),
        f1=(
            "f1",
            "mean"
        ),
        mcc=(
            "mcc",
            "mean"
        )
    )
    .reset_index()
)


plt.figure(
    figsize=(12, 7)
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "accuracy"
    ],
    label="Accuracy"
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "precision"
    ],
    label="Precision"
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "recall"
    ],
    label="Recall"
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "f1"
    ],
    linewidth=3,
    label="F1"
)


plt.plot(
    mean_threshold_results[
        "threshold"
    ],
    mean_threshold_results[
        "specificity"
    ],
    label="Specificity"
)


plt.axvline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Final threshold = "
        f"{FINAL_THRESHOLD:.3f}"
    )
)


plt.xlabel(
    "Cosine Similarity Threshold"
)


plt.ylabel(
    "Metric"
)


plt.title(
    "Threshold Selection — Cross-Validated Metrics"
)


plt.legend(
    bbox_to_anchor=(
        1.02,
        1
    ),
    loc="upper left"
)


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "03_threshold_vs_metrics.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 47. PLOT 4 — THRESHOLD STABILITY
# ============================================================

plt.figure(
    figsize=(9, 6)
)


plt.plot(
    fold_results_df[
        "fold"
    ],
    fold_results_df[
        "threshold"
    ],
    marker="o",
    linewidth=2
)


plt.axhline(
    FINAL_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Median threshold = "
        f"{FINAL_THRESHOLD:.3f}"
    )
)


plt.xlabel(
    "Fold"
)


plt.ylabel(
    "Best Cosine Threshold"
)


plt.title(
    "Threshold Stability Across Folds"
)


plt.xticks(
    fold_results_df[
        "fold"
    ]
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "04_threshold_stability.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 48. PLOT 5 — ROC CURVE
# ============================================================

fpr_curve, tpr_curve, roc_thresholds = (
    roc_curve(
        oof_y_true,
        oof_similarity
    )
)


plt.figure(
    figsize=(7, 7)
)


plt.plot(
    fpr_curve,
    tpr_curve,
    linewidth=3,
    label=(
        f"ResNet50 "
        f"(AUC = {roc_auc:.4f})"
    )
)


plt.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    label="Random"
)


plt.xlabel(
    "False Positive Rate"
)


plt.ylabel(
    "True Positive Rate"
)


plt.title(
    "ROC Curve — ResNet50"
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "05_roc_curve.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 49. PLOT 6 — PRECISION RECALL CURVE
# ============================================================

precision_curve, recall_curve, pr_thresholds = (
    precision_recall_curve(
        oof_y_true,
        oof_similarity
    )
)


plt.figure(
    figsize=(7, 7)
)


plt.plot(
    recall_curve,
    precision_curve,
    linewidth=3,
    label=(
        f"PR-AUC = "
        f"{pr_auc:.4f}"
    )
)


plt.xlabel(
    "Recall"
)


plt.ylabel(
    "Precision"
)


plt.title(
    "Precision-Recall Curve — ResNet50"
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "06_precision_recall_curve.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 50. PLOT 7 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    oof_y_true,
    oof_prediction,
    labels=[
        0,
        1
    ]
)


plt.figure(
    figsize=(7, 6)
)


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Different",
        "Same"
    ],
    yticklabels=[
        "Different",
        "Same"
    ]
)


plt.xlabel(
    "Predicted"
)


plt.ylabel(
    "Actual"
)


plt.title(
    "Confusion Matrix — ResNet50"
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "07_confusion_matrix.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 51. PLOT 8 — FOLD METRIC COMPARISON
# ============================================================

plot_metrics = [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc"
]


plt.figure(
    figsize=(12, 7)
)


for metric in plot_metrics:

    plt.plot(
        fold_results_df[
            "fold"
        ],
        fold_results_df[
            metric
        ],
        marker="o",
        label=metric.upper()
    )


plt.xlabel(
    "Fold"
)


plt.ylabel(
    "Score"
)


plt.title(
    "ResNet50 Cross-Validation Metric Stability"
)


plt.xticks(
    fold_results_df[
        "fold"
    ]
)


plt.legend(
    bbox_to_anchor=(
        1.02,
        1
    ),
    loc="upper left"
)


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "08_cv_metric_stability.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 52. RETRIEVAL FUNCTION
# ============================================================
#
# Query:
#
#       image
#        ↓
#       ResNet embedding
#        ↓
#       cosine similarity against all images
#        ↓
#       sort descending
#        ↓
#       top-K
#
# ============================================================

def get_resnet_nearest_neighbors(
    dataframe,
    embeddings,
    query_idx,
    top_k=10
):

    query_embedding = (
        embeddings[
            query_idx
        ]
    )


    # Because embeddings are L2 normalized:
    #
    # cosine similarity = dot product
    #

    similarities = (
        embeddings
        @
        query_embedding
    )


    # --------------------------------------------------------
    # Exclude query itself
    # --------------------------------------------------------

    similarities[
        query_idx
    ] = -np.inf


    # --------------------------------------------------------
    # Sort descending
    # --------------------------------------------------------

    nearest_indices = np.argsort(
        -similarities
    )[
        :top_k
    ]


    result = []


    for rank, idx in enumerate(
        nearest_indices,
        start=1
    ):

        result.append({

            "rank": rank,

            "query_idx": query_idx,

            "neighbor_idx": int(idx),

            "similarity":
                float(
                    similarities[idx]
                ),

            "query_product":
                dataframe.loc[
                    query_idx,
                    PRODUCT_ID_COLUMN
                ],

            "neighbor_product":
                dataframe.loc[
                    idx,
                    PRODUCT_ID_COLUMN
                ],

            "neighbor_image":
                dataframe.loc[
                    idx,
                    IMAGE_COLUMN
                ],

            "neighbor_image_path":
                dataframe.loc[
                    idx,
                    "image_path"
                ]

        })


    return pd.DataFrame(
        result
    )


# ============================================================
# 53. RETRIEVAL METRICS
# ============================================================
#
# Top-1:
#
#       Is the nearest image from the same product?
#
# Top-K:
#
#       Does at least one of the top-K images belong to
#       the same product?
#
# ============================================================

def calculate_retrieval_metrics(
    dataframe,
    embeddings,
    top_k=10
):

    top1_correct = 0

    topk_correct = 0

    evaluated = 0


    for query_idx in tqdm(
        dataframe.index,
        desc="Retrieval evaluation"
    ):

        query_product = dataframe.loc[
            query_idx,
            PRODUCT_ID_COLUMN
        ]


        neighbors = (
            get_resnet_nearest_neighbors(
                dataframe,
                embeddings,
                query_idx,
                top_k=top_k
            )
        )


        if len(neighbors) == 0:

            continue


        evaluated += 1


        # ----------------------------------------------------
        # Top-1
        # ----------------------------------------------------

        if (
            neighbors.iloc[
                0
            ][
                "neighbor_product"
            ]
            ==
            query_product
        ):

            top1_correct += 1


        # ----------------------------------------------------
        # Top-K
        # ----------------------------------------------------

        if (
            query_product
            in
            neighbors[
                "neighbor_product"
            ].values
        ):

            topk_correct += 1


    return {

        "queries": evaluated,

        "top_1_accuracy":
            top1_correct / evaluated
            if evaluated > 0
            else 0,

        f"top_{top_k}_hit_rate":
            topk_correct / evaluated
            if evaluated > 0
            else 0

    }


# ============================================================
# 54. CALCULATE RETRIEVAL PERFORMANCE
# ============================================================

retrieval_result = calculate_retrieval_metrics(
    validation_df,
    validation_embeddings,
    top_k=TOP_K
)


print("\n" + "=" * 90)

print("RESNET50 RETRIEVAL RESULTS")

print("=" * 90)


print(
    f"Queries:"
    f" {retrieval_result['queries']:,}"
)


print(
    f"Top-1 Accuracy:"
    f" {retrieval_result['top_1_accuracy']:.4f}"
)


print(
    f"Top-{TOP_K} Hit Rate:"
    f" {retrieval_result[f'top_{TOP_K}_hit_rate']:.4f}"
)


# ============================================================
# 55. SAMPLE QUERY
# ============================================================

sample_query_idx = (
    validation_df.index[0]
)


sample_neighbors = (
    get_resnet_nearest_neighbors(
        validation_df,
        validation_embeddings,
        sample_query_idx,
        top_k=TOP_K
    )
)


print("\n" + "=" * 90)

print("SAMPLE NEAREST NEIGHBOURS")

print("=" * 90)


print(
    "\nQuery image:"
)


print(
    validation_df.loc[
        sample_query_idx
    ][
        [
            IMAGE_COLUMN,
            IMAGE_ID_COLUMN,
            PRODUCT_ID_COLUMN
        ]
    ]
    if IMAGE_ID_COLUMN in validation_df.columns
    else validation_df.loc[
        sample_query_idx
    ][
        [
            IMAGE_COLUMN,
            PRODUCT_ID_COLUMN
        ]
    ]
)


print(
    "\nNearest neighbours:"
)


print(
    sample_neighbors[
        [
            "rank",
            "neighbor_image",
            "neighbor_product",
            "similarity"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 56. SAVE SAMPLE RETRIEVAL
# ============================================================

sample_neighbors.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sample_nearest_neighbors.csv"
    ),
    index=False
)


# ============================================================
# 57. RETRIEVAL TOP-K CURVE
# ============================================================
#
# Calculate:
#
# Top-1
# Top-2
# ...
# Top-10
#
# ============================================================

retrieval_curve = []


for k in range(
    1,
    TOP_K + 1
):

    hits = 0

    evaluated = 0


    for query_idx in validation_df.index:

        query_product = validation_df.loc[
            query_idx,
            PRODUCT_ID_COLUMN
        ]


        neighbors = (
            get_resnet_nearest_neighbors(
                validation_df,
                validation_embeddings,
                query_idx,
                top_k=k
            )
        )


        if len(neighbors) == 0:

            continue


        evaluated += 1


        if (
            query_product
            in
            neighbors[
                "neighbor_product"
            ].values
        ):

            hits += 1


    hit_rate = (
        hits / evaluated
        if evaluated > 0
        else 0
    )


    retrieval_curve.append({

        "k": k,

        "hit_rate": hit_rate

    })


retrieval_curve_df = pd.DataFrame(
    retrieval_curve
)


plt.figure(
    figsize=(9, 6)
)


plt.plot(
    retrieval_curve_df[
        "k"
    ],
    retrieval_curve_df[
        "hit_rate"
    ],
    marker="o",
    linewidth=2
)


plt.xlabel(
    "K"
)


plt.ylabel(
    "Hit Rate"
)


plt.title(
    "ResNet50 Top-K Retrieval Performance"
)


plt.xticks(
    range(
        1,
        TOP_K + 1
    )
)


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "09_top_k_retrieval.png"
    ),
    dpi=200
)


plt.show()


retrieval_curve_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "retrieval_top_k_curve.csv"
    ),
    index=False
)


# ============================================================
# 58. ERROR ANALYSIS
# ============================================================
#
# FALSE POSITIVE:
#
# Different product
# BUT
# high cosine similarity
#
#
# FALSE NEGATIVE:
#
# Same product
# BUT
# low cosine similarity
#
# ============================================================

false_positives = oof_predictions[
    (
        oof_predictions[
            "label"
        ] == 0
    )
    &
    (
        oof_predictions[
            "final_prediction"
        ] == 1
    )
].copy()


false_negatives = oof_predictions[
    (
        oof_predictions[
            "label"
        ] == 1
    )
    &
    (
        oof_predictions[
            "final_prediction"
        ] == 0
    )
].copy()


print("\n" + "=" * 90)

print("ERROR ANALYSIS")

print("=" * 90)


print(
    f"False Positives:"
    f" {len(false_positives):,}"
)


print(
    f"False Negatives:"
    f" {len(false_negatives):,}"
)


# ============================================================
# 59. MOST CONFIDENT FALSE POSITIVES
# ============================================================
#
# Highest cosine similarity among wrong negatives.
#
# ============================================================

false_positives = false_positives.sort_values(
    "cosine_similarity",
    ascending=False
)


print(
    "\nMost visually similar FALSE POSITIVES:"
)


fp_columns = [

    "image_1",

    "image_2",

    "product_id_1",

    "product_id_2",

    "cosine_similarity",

    "label",

    "final_prediction"

]


fp_columns = [
    column
    for column in fp_columns
    if column in false_positives.columns
]


print(
    false_positives[
        fp_columns
    ].head(25).to_string(
        index=False
    )
)


# ============================================================
# 60. MOST DIFFICULT FALSE NEGATIVES
# ============================================================
#
# Lowest cosine similarity among same-product pairs.
#
# ============================================================

false_negatives = false_negatives.sort_values(
    "cosine_similarity",
    ascending=True
)


print(
    "\nMost visually dissimilar FALSE NEGATIVES:"
)


fn_columns = [

    "image_1",

    "image_2",

    "product_id_1",

    "product_id_2",

    "cosine_similarity",

    "label",

    "final_prediction"

]


fn_columns = [
    column
    for column in fn_columns
    if column in false_negatives.columns
]


print(
    false_negatives[
        fn_columns
    ].head(25).to_string(
        index=False
    )
)


# ============================================================
# 61. SAVE ERROR ANALYSIS
# ============================================================

false_positives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "false_positives.csv"
    ),
    index=False
)


false_negatives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "false_negatives.csv"
    ),
    index=False
)


# ============================================================
# 62. ERROR DISTRIBUTION
# ============================================================

error_summary = pd.DataFrame({

    "error_type": [

        "True Negative",

        "False Positive",

        "False Negative",

        "True Positive"

    ],

    "count": [

        tn,

        fp,

        fn,

        tp

    ]

})


plt.figure(
    figsize=(9, 6)
)


sns.barplot(
    data=error_summary,
    x="error_type",
    y="count"
)


plt.xlabel(
    "Prediction Category"
)


plt.ylabel(
    "Number of Pairs"
)


plt.title(
    "ResNet50 Prediction Breakdown"
)


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "10_prediction_breakdown.png"
    ),
    dpi=200
)


plt.show()


# ============================================================
# 63. COMPARE SAME-PRODUCT VS DIFFERENT-PRODUCT MEANS
# ============================================================

class_similarity_summary = (
    pairs_df
    .groupby(
        "label"
    )[
        "cosine_similarity"
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max"
        ]
    )
)


class_similarity_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "similarity_class_statistics.csv"
    )
)


print(
    "\nSimilarity statistics:"
)


print(
    class_similarity_summary
)


# ============================================================
# 64. FINAL CONFIGURATION / EXPERIMENT RECORD
# ============================================================

experiment_configuration = pd.DataFrame({

    "parameter": [

        "Approach",

        "Image representation",

        "CNN",

        "Pretrained weights",

        "CNN trained on Shopee data",

        "Embedding dimension",

        "Embedding normalization",

        "Similarity",

        "Threshold learning",

        "Cross-validation folds",

        "Validation size",

        "Positive pairs per product",

        "Negative pairs",

        "Final threshold",

        "Accuracy",

        "Balanced Accuracy",

        "Precision",

        "Recall",

        "Specificity",

        "FPR",

        "FNR",

        "F1",

        "MCC",

        "ROC-AUC",

        "PR-AUC",

        "Top-1 retrieval",

        f"Top-{TOP_K} retrieval"

    ],


    "value": [

        "Pretrained CNN image matching",

        "ResNet50 global image embedding",

        "ResNet50",

        "ImageNet pretrained",

        "No",

        2048,

        "L2 normalization",

        "Cosine similarity",

        "Maximum F1 on training folds",

        N_SPLITS,

        VALIDATION_SIZE,

        POSITIVE_PAIRS_PER_PRODUCT,

        NEGATIVE_PAIRS,

        FINAL_THRESHOLD,

        final_accuracy,

        final_balanced_accuracy,

        final_precision,

        final_recall,

        final_specificity,

        final_fpr,

        final_fnr,

        final_f1,

        final_mcc,

        roc_auc,

        pr_auc,

        retrieval_result[
            "top_1_accuracy"
        ],

        retrieval_result[
            f"top_{TOP_K}_hit_rate"
        ]

    ]

})


experiment_configuration.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "experiment_configuration.csv"
    ),
    index=False
)


# ============================================================
# 65. FINAL RESULTS SUMMARY
# ============================================================

print("\n" + "=" * 90)

print("FINAL RESNET50 APPROACH SUMMARY")

print("=" * 90)


print(
    f"""
DATASET
-------
Original rows:
    {len(df):,}

Training rows:
    {len(train_df):,}

Validation rows:
    {len(validation_df):,}

Training products:
    {train_df[PRODUCT_ID_COLUMN].nunique():,}

Validation products:
    {validation_df[PRODUCT_ID_COLUMN].nunique():,}


PAIR DATASET
------------
Positive pairs:
    {len(positive_pairs):,}

Negative pairs:
    {len(negative_pairs):,}

Total pairs:
    {len(pairs_df):,}


VISION MODEL
------------
Model:
    ResNet50

Pretrained:
    ImageNet

CNN training:
    Frozen / no fine-tuning

Embedding dimension:
    2048

Normalization:
    L2


SIMILARITY
----------
Metric:
    Cosine similarity

Decision:
    similarity >= threshold -> MATCH
    similarity < threshold  -> NON-MATCH


CROSS-VALIDATION
----------------
Folds:
    {N_SPLITS}

Fold thresholds:
    {fold_results_df['threshold'].round(4).tolist()}

Final threshold:
    {FINAL_THRESHOLD:.4f}


CLASSIFICATION
---------------
Accuracy:
    {final_accuracy:.4f}

Balanced Accuracy:
    {final_balanced_accuracy:.4f}

Precision:
    {final_precision:.4f}

Recall:
    {final_recall:.4f}

Specificity:
    {final_specificity:.4f}

FPR:
    {final_fpr:.4f}

FNR:
    {final_fnr:.4f}

F1:
    {final_f1:.4f}

MCC:
    {final_mcc:.4f}

ROC-AUC:
    {roc_auc:.4f}

PR-AUC:
    {pr_auc:.4f}


CONFUSION MATRIX
----------------
TP:
    {tp}

TN:
    {tn}

FP:
    {fp}

FN:
    {fn}


RETRIEVAL
---------
Top-1:
    {retrieval_result['top_1_accuracy']:.4f}

Top-{TOP_K}:
    {retrieval_result[f'top_{TOP_K}_hit_rate']:.4f}


OUTPUT
------
{OUTPUT_DIR}
"""
)


# ============================================================
# 66. SAVE RETRIEVAL SUMMARY
# ============================================================

retrieval_summary = pd.DataFrame({

    "metric": [

        "Top-1 Accuracy",

        f"Top-{TOP_K} Hit Rate"

    ],

    "value": [

        retrieval_result[
            "top_1_accuracy"
        ],

        retrieval_result[
            f"top_{TOP_K}_hit_rate"
        ]

    ]

})


retrieval_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "retrieval_summary.csv"
    ),
    index=False
)


# ============================================================
# 67. LIST ALL GENERATED FILES
# ============================================================

print("\n" + "=" * 90)

print("GENERATED FILES")

print("=" * 90)


for filename in sorted(
    os.listdir(
        OUTPUT_DIR
    )
):

    print(
        "  ",
        filename
    )


print("\n" + "=" * 90)

print("RESNET50 APPROACH COMPLETE")

print("=" * 90)